<a href="https://colab.research.google.com/github/tinawanggg/Wang_DSPN_S26/blob/master/book/exercises/15_power-analysis-via-simulations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 15: Power analyses

This  assignment is designed to give you practice with Monte Carlo methods to conduct power analyses via simulation. You won't need to load in any data for this homework. We will, however, be using parts of the homework from last week.

---
## 1. Simulating data (1 point)


Pull your `simulate_data()` function from your last homework and add it below.

As a reminder, this function simulates the relationship between age, word reading experience, and reading comprehension skill.

`c` is reading comprehension, and `x` is word reading experience.

In [4]:
sample_size = 100 # How many children in data set?
age_lo = 80     # minimum age, in months
age_hi = 200    # maximum age, in months
beta_xa = 0.5   # amount by which experience changes for increase of one month in age
beta_x0 = -5    # amount of experience when age = 0 (not interpretable, since minimum age for this data is 80 months)
sd_x = 50       # standard dev of gaussian noise term, epsilon_x
beta_ca = 0.8   # amount that comprehension score improves for every increase of one unit in age
beta_cx = 3     # amount that comprehension score improves for every increase of one unit in reading experience
beta_c0 = 10    # comprehension score when reading experience is 0.
sd_c = 85      # standard dev of gaussian noise term, epsilon_c

simulate_data <- function(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
      age <- runif(sample_size, min = age_lo, max = age_hi)

      epsilon_x <- rnorm(sample_size, mean = 0, sd = sd_x)
      x <- beta_xa * age + beta_x0 + epsilon_x

      epsilon_c <- rnorm(sample_size, mean = 0, sd = sd_c)
      c <- beta_ca * age + beta_cx * x + beta_c0 + epsilon_c

      return(data.frame(age=age,x=x,c=c)) # it's actually bad form to have a variable named "c" in R, my bad...
}

dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
head(dat)

,age,x,c
,<dbl>,<dbl>,<dbl>
1,114.44869,78.95677,203.4085
2,161.92092,56.20465,493.8533
3,131.65203,83.31063,423.0443
4,165.14054,98.74040,556.0298
5,189.87452,153.00823,652.0244
6,99.98367,-64.76893,-79.4036


---
## 2. `run_analysis()` function (2 points)

Last week, we looked at whether word reading experience(`x`) mediated the relation between `age` and reading comprehension (`c`).

Now we're going to use our `simulate_data()` function to conduct a power analysis. The goal is to determine how many participants we would need in order to detect both the mediated and the direct effects in this data.

*Note: We're going to pretend for the sake of simplicity that we don't have any control over the ages of the children we get (so ages are generated using `runif(sample_size, age_lo, age_hi)`, although of course this would be an unusual situation in reality.*

First, write a function, `run_analysis()`, that takes in simulated data, runs **your mediation from last week**, and returns a vector containing the ACME and ADE estimates and p-values (these are the `d0`, `d0.p`, `z0`, and `z0.p` features of the mediated model object, e.g., `fitMed$d0.p`). Print this function's output for the data we simulated previously.

In [6]:
# WRITE YOUR CODE HERE
library(tidyverse)
install.packages("mediation")
library(mediation)

run_analysis <- function(dat) {
  model_m <- lm(x ~ age, data = dat)
  model_y <- lm(c ~ age + x, data = dat)
  med_out <- mediate(model_m, model_y, treat = "age", mediator = "x")

  return(c(
    d0   = med_out$d0,
    d0.p = med_out$d0.p,
    z0   = med_out$z0,
    z0.p = med_out$z0.p
  ))
}

run_analysis(dat)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



d0      d0.p        z0      z0.p 
1.5258817 0.0000000 0.9154407 0.0020000

---
## 3. `repeat_analysis()` function (3 points)

Next fill in the function `repeat_analysis()` below so that it simulates and analyzes data `num_simulations` times. Store the outputs from each simulation in the `simouts` matrix. Calculate and return the coverage across all the simulations run for both ACME and ADE.

In [7]:
repeat_analysis <- function(num_simulations, alpha, sample_size, age_lo, age_hi,
        beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
    # Initialize simouts matrix for storing each output from run_analysis()
    simouts <- matrix(rep(NA, num_simulations*4), nrow=num_simulations, ncol=4)

    # Start simulating
    for (i in 1:num_simulations) {
      dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
      simouts[i,] <- run_analysis(dat)
    }

    # Calculate coverage for both ACME and ADE estimates using p-values in simouts
    ACME_cov = mean(simouts[,2] < alpha)
    ADE_cov = mean(simouts[,4] < alpha)

    return(list(ACME_cov = ACME_cov, ADE_cov = ADE_cov))
}

Now run the `repeat_analysis()` function using the same parameter settings as above, for 10 simulations, with an alpha criterion of 0.01.

In [8]:
# WRITE YOUR CODE HERE
repeat_analysis(num_simulations = 10, alpha = 0.01, sample_size = sample_size,
                age_lo = age_lo, age_hi = age_hi, beta_xa = beta_xa,
                beta_x0 = beta_x0, sd_x = sd_x, beta_ca = beta_ca,
                beta_cx = beta_cx, beta_c0 = beta_c0, sd_c = sd_c)

$ACME_cov
[1] 1

$ADE_cov
[1] 0.7

---
## 4. Testing different sample sizes (2 points)

Finally, do the same thing (10 simulations, alpha criterion of 0.01) but for 5 different sample sizes: 50, 75, 100, 125, 150. You can do this using `map` (as in the tutorial), or a simple `for` loop, or by calculating each individually. Up to you! This should take around 3 minutes to run.

In [10]:
# WRITE YOUR CODE HERE
sample_sizes <- c(50, 75, 100, 125, 150)

results <- map(sample_sizes, ~repeat_analysis(num_simulations = 10, alpha = 0.01,
                sample_size = .x, age_lo = age_lo, age_hi = age_hi,
                beta_xa = beta_xa, beta_x0 = beta_x0, sd_x = sd_x,
                beta_ca = beta_ca, beta_cx = beta_cx, beta_c0 = beta_c0, sd_c = sd_c))

Print your results.

In [11]:
# WRITE YOUR CODE HERE
results

[[1]]
[[1]]$ACME_cov
[1] 0.3

[[1]]$ADE_cov
[1] 0.3


[[2]]
[[2]]$ACME_cov
[1] 0.5

[[2]]$ADE_cov
[1] 0.4


[[3]]
[[3]]$ACME_cov
[1] 0.9

[[3]]$ADE_cov
[1] 0.6


[[4]]
[[4]]$ACME_cov
[1] 0.8

[[4]]$ADE_cov
[1] 0.4


[[5]]
[[5]]$ACME_cov
[1] 1

[[5]]$ADE_cov
[1] 0.8

## 5. Reflection (2 pts)

If this were a real power analysis, we'd want to run more simulations per sample size (to get a more precise estimate of power) and we may also want to test out some other values of the parameters we used to simulate our data. However, what would you conclude just based on the results above?

> *From the results, it is observed that power for detecting both the ACME and ADE generally increases with increased sample size. The ACME (mediated effect through word reading experience) is detected more reliably than the ADE (direct effect of age on comprehension) across nearly all sample sizes. The ADE appears to require a larger sample size to detect consistently.*
>

Given how we generated the data, why was the direct effect harder to detect than the mediated effect?
> *This is due to how the betas were set up: beta_cx = 3 is quite large in comparison to beta_ca = 0.8, which insinuates that word reading experience has a stronger influence on comprehension than direct effect of age.*

**DUE:** 11:59pm EST, March 31, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *N/A*

**GenAI Utilization** Did you utilize any generative AI tools on this assignment? If so, please list the item and the paste respective prompt you used.

>
>